# Three-House Demand and Solar Forecasting

This notebook contains the complete workflow: inspect and prepare the local Ausgrid CSV, create hourly demand and solar data with cyclical calendar features (hour_sin, hour_cos, dow_sin, dow_cos) for three houses, make 24-hour sequences, split chronologically, fit a shared two-output LSTM, evaluate against a persistence baseline, and save the requested artifacts. It does not use electricity price as a model input. Test scores are historical held-out evaluation, not future forecasts.

**Run all cells from top to bottom.** No standalone training script is required.

In [1]:
from pathlib import Path
import random

import joblib
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import MinMaxScaler

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
RAW_PATH = PROJECT_ROOT / "data" / "raw" / "Solar home 2012-2013.csv"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
HOURLY_PATH = OUTPUT_DIR / "three_house_hourly_lstm_ready.csv"
MAPPING_PATH = OUTPUT_DIR / "customer_id_mapping.csv"

LOOKBACK_HOURS = 24
TRAIN_FRACTION = 0.80
VALIDATION_FRACTION_OF_TRAIN = 0.10
FEATURE_COLUMNS = [
    "total_demand_kwh",
    "solar_generation_kwh",
    "hour_sin",
    "hour_cos",
    "dow_sin",
    "dow_cos",
]
TARGET_COLUMNS = ["total_demand_kwh", "solar_generation_kwh"]
RANDOM_SEED = 42
tf.keras.utils.set_random_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

# Ausgrid file row 1 is a title; row 2 is the real header. Preserve literal
# "NA" and blank Row Quality values separately (blank means actual readings).
raw = pd.read_csv(RAW_PATH, header=1, low_memory=False, keep_default_na=False)
interval_columns = raw.columns[5:53].tolist()
if len(interval_columns) != 48 or raw.columns[53] != "Row Quality":
    raise ValueError("Source CSV does not have the documented 48-interval schema.")
raw["Customer"] = pd.to_numeric(raw["Customer"], errors="coerce")
raw["Generator Capacity"] = pd.to_numeric(raw["Generator Capacity"], errors="coerce")
raw["date"] = pd.to_datetime(raw["date"], dayfirst=True, errors="coerce")
raw[interval_columns] = raw[interval_columns].apply(pd.to_numeric, errors="coerce")
if raw[["Customer", "Generator Capacity", "date"]].isna().any().any():
    raise ValueError("Invalid customer, capacity, or date in source.")
if raw.duplicated(["Customer", "Consumption Category", "date"]).any():
    raise ValueError("Duplicate customer/category/date records in source.")

source_dates = sorted(raw["date"].unique())
if len(source_dates) != 365:
    raise ValueError(f"Expected 365 dates; found {len(source_dates)}.")
actual = raw.loc[
    raw["Row Quality"].eq("") & raw[interval_columns].notna().all(axis=1)
].copy()
coverage = actual.groupby(["Customer", "Consumption Category"])["date"].nunique().unstack(fill_value=0)
required_categories = ["GC", "CL", "GG"]
coverage = coverage.reindex(columns=required_categories, fill_value=0)
eligible_customers = sorted(
    int(customer)
    for customer in coverage.index[(coverage[required_categories] == 365).all(axis=1)]
)
if len(eligible_customers) < 3:
    raise ValueError("Fewer than three houses have full-year actual GC, CL and GG records.")
selected_customers = eligible_customers[:3]
house_ids = {customer: f"H{i:03d}" for i, customer in enumerate(selected_customers, 1)}

selected = actual.loc[actual["Customer"].isin(selected_customers)]
capacity_by_customer = selected.drop_duplicates("Customer").set_index("Customer")["Generator Capacity"].to_dict()
postcode_by_customer = selected.drop_duplicates("Customer").set_index("Customer")["Postcode"].to_dict()
keyed = selected.set_index(["Customer", "date", "Consumption Category"])
hourly_records = []
for customer in selected_customers:
    for day in source_dates:
        category_rows = {
            category: keyed.loc[(customer, day, category), interval_columns]
            for category in required_categories
        }
        for hour in range(24):
            first_half, second_half = interval_columns[2 * hour:2 * hour + 2]
            gc_kwh = float(category_rows["GC"][first_half] + category_rows["GC"][second_half])
            cl_kwh = float(category_rows["CL"][first_half] + category_rows["CL"][second_half])
            gg_kwh = float(category_rows["GG"][first_half] + category_rows["GG"][second_half])
            timestamp = pd.Timestamp(day) + pd.Timedelta(hours=hour)
            hour_val = timestamp.hour
            dow_val = timestamp.dayofweek
            hourly_records.append({
                "house_id": house_ids[customer],
                "timestamp": timestamp,
                "total_demand_kwh": gc_kwh + cl_kwh,
                "solar_generation_kwh": gg_kwh,
                "generator_capacity_kwp": float(capacity_by_customer[customer]),
                "hour": hour_val,
                "day_of_week": dow_val,
                "month": timestamp.month,
                "hour_sin": float(np.sin(2 * np.pi * hour_val / 24.0)),
                "hour_cos": float(np.cos(2 * np.pi * hour_val / 24.0)),
                "dow_sin": float(np.sin(2 * np.pi * dow_val / 7.0)),
                "dow_cos": float(np.cos(2 * np.pi * dow_val / 7.0)),
            })

hourly = pd.DataFrame(hourly_records).sort_values(["house_id", "timestamp"]).reset_index(drop=True)
house_mapping = pd.DataFrame([
    {
        "house_id": house_ids[customer],
        "source_customer_id": customer,
        "postcode": postcode_by_customer[customer],
        "generator_capacity_kwp": capacity_by_customer[customer],
        "selection_criteria": "365 complete actual days for GC, CL, and GG",
    }
    for customer in selected_customers
])
assert len(hourly) == 3 * 365 * 24
assert not hourly.duplicated(["house_id", "timestamp"]).any()
assert hourly[["total_demand_kwh", "solar_generation_kwh"]].notna().all().all()
hourly.to_csv(HOURLY_PATH, index=False)
house_mapping.to_csv(MAPPING_PATH, index=False)

print(f"Source shape: {raw.shape}; customers: {raw['Customer'].nunique()}; eligible full-year houses: {len(eligible_customers)}")
print("Selected customer mapping:")
display(house_mapping)
print(f"Created {len(hourly):,} hourly rows at {HOURLY_PATH}")
display(hourly.head())

/Users/dimpalkava/Stochastic-Game-Theoretic-Optimization-under-uncertain-energy-conditions/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Source shape: (268557, 54); customers: 300; eligible full-year houses: 117
Selected customer mapping:


,house_id,source_customer_id,postcode,generator_capacity_kwp,selection_criteria
0,H001,1,2076,3.78,"365 complete actual days for GC, CL, and GG"
1,H002,3,2263,1.00,"365 complete actual days for GC, CL, and GG"
2,H003,4,2220,1.00,"365 complete actual days for GC, CL, and GG"


Created 26,280 hourly rows at /Users/dimpalkava/Stochastic-Game-Theoretic-Optimization-under-uncertain-energy-conditions/data/processed/three_house_hourly_lstm_ready.csv


,house_id,timestamp,total_demand_kwh,solar_generation_kwh,generator_capacity_kwp,hour,day_of_week,month,hour_sin,hour_cos,dow_sin,dow_cos
0,H001,2012-07-01 00:00:00,4.141,0.0,3.78,0,6,7,0.000000,1.000000,-0.781831,0.62349
1,H001,2012-07-01 01:00:00,3.661,0.0,3.78,1,6,7,0.258819,0.965926,-0.781831,0.62349
2,H001,2012-07-01 02:00:00,1.909,0.0,3.78,2,6,7,0.500000,0.866025,-0.781831,0.62349
3,H001,2012-07-01 03:00:00,2.466,0.0,3.78,3,6,7,0.707107,0.707107,-0.781831,0.62349
4,H001,2012-07-01 04:00:00,1.166,0.0,3.78,4,6,7,0.866025,0.500000,-0.781831,0.62349


## 1. Make 24-hour sequences without crossing houses

For each house independently, each input is the preceding 24 observed hours and its features have shape `(24, 6)` (`total_demand_kwh`, `solar_generation_kwh`, `hour_sin`, `hour_cos`, `dow_sin`, `dow_cos`). The target is the next hour's `[total_demand_kwh, solar_generation_kwh]`, shape `(2,)`. The first 80% of each house's chronology is the development period; its final 20% is held out for testing. The final 10% within development is used as chronological validation. The first test input may use the final 24 training observations as context; later one-step test predictions use the preceding observed hours. Test targets are never training targets.

In [2]:
houses = {
    house_id: frame.sort_values("timestamp").reset_index(drop=True)
    for house_id, frame in hourly.groupby("house_id", sort=True)
}
n_rows = {len(frame) for frame in houses.values()}
assert len(n_rows) == 1
n_rows = n_rows.pop()
split_index = int(n_rows * TRAIN_FRACTION)
validation_start = int(split_index * (1 - VALIDATION_FRACTION_OF_TRAIN))

# Fit the shared scalers on model-fit rows only. Validation and test observations
# are deliberately excluded from scaler fitting.
feature_scaler_fit_values = np.concatenate([
    frame.loc[: validation_start - 1, FEATURE_COLUMNS].to_numpy(dtype=np.float64)
    for frame in houses.values()
], axis=0)
target_scaler_fit_values = np.concatenate([
    frame.loc[: validation_start - 1, TARGET_COLUMNS].to_numpy(dtype=np.float64)
    for frame in houses.values()
], axis=0)

feature_scaler = MinMaxScaler(feature_range=(0, 1))
feature_scaler.fit(feature_scaler_fit_values)

target_scaler = MinMaxScaler(feature_range=(0, 1))
target_scaler.fit(target_scaler_fit_values)
scaler = target_scaler
scaler.feature_scaler = feature_scaler

sequence_lists = {
    name: [] for name in ["X_train", "y_train", "X_val", "y_val", "X_test", "y_test", "baseline_test"]
}
sequence_meta = {
    name: [] for name in [
        "train_house_id", "train_target_timestamp", "train_target_row_index",
        "val_house_id", "val_target_timestamp", "val_target_row_index",
        "test_house_id", "test_target_timestamp", "test_target_row_index", "test_actual_raw",
    ]
}

for house_id, frame in houses.items():
    raw_feats = frame[FEATURE_COLUMNS].to_numpy(dtype=np.float64)
    scaled_feats = feature_scaler.transform(raw_feats)
    raw_targets = frame[TARGET_COLUMNS].to_numpy(dtype=np.float64)
    scaled_targets = target_scaler.transform(raw_targets)
    timestamps = frame["timestamp"].to_numpy(dtype="datetime64[ns]")
    for target_idx in range(LOOKBACK_HOURS, n_rows):
        x_raw = raw_feats[target_idx - LOOKBACK_HOURS:target_idx]
        x_scaled = scaled_feats[target_idx - LOOKBACK_HOURS:target_idx]
        y_raw = raw_targets[target_idx]
        y_scaled = scaled_targets[target_idx]
        if target_idx < validation_start:
            part = "train"
        elif target_idx < split_index:
            part = "val"
        else:
            part = "test"
        sequence_lists[f"X_{part}"].append(x_scaled.astype(np.float32))
        sequence_lists[f"y_{part}"].append(y_scaled.astype(np.float32))
        sequence_meta[f"{part}_house_id"].append(house_id)
        sequence_meta[f"{part}_target_timestamp"].append(timestamps[target_idx])
        sequence_meta[f"{part}_target_row_index"].append(target_idx)
        if part == "test":
            sequence_lists["baseline_test"].append(x_raw[-1, :len(TARGET_COLUMNS)].astype(np.float32))
            sequence_meta["test_actual_raw"].append(y_raw.astype(np.float32))

seq = {name: np.asarray(values, dtype=np.float32) for name, values in sequence_lists.items()}
for key, values in sequence_meta.items():
    if key.endswith("_timestamp"):
        seq[key] = np.asarray(values, dtype="datetime64[ns]")
    elif key.endswith("_house_id"):
        seq[key] = np.asarray(values, dtype="U4")
    else:
        seq[key] = np.asarray(values)
seq["feature_columns"] = np.asarray(FEATURE_COLUMNS, dtype="U32")
seq["target_columns"] = np.asarray(TARGET_COLUMNS, dtype="U32")
seq["lookback_hours"] = np.asarray(LOOKBACK_HOURS)
seq["house_train_test_split_index"] = np.asarray(split_index)
seq["scaler_fit_end_index"] = np.asarray(validation_start - 1)

assert seq["X_train"].shape[1:] == (LOOKBACK_HOURS, len(FEATURE_COLUMNS)) and seq["y_train"].shape[1:] == (len(TARGET_COLUMNS),)
assert seq["X_val"].shape[1:] == (LOOKBACK_HOURS, len(FEATURE_COLUMNS)) and seq["y_val"].shape[1:] == (len(TARGET_COLUMNS),)
assert seq["X_test"].shape[1:] == (LOOKBACK_HOURS, len(FEATURE_COLUMNS)) and seq["y_test"].shape[1:] == (len(TARGET_COLUMNS),)
assert seq["train_target_row_index"].max() < split_index
assert seq["test_target_row_index"].min() == split_index
assert np.all(seq["test_target_row_index"] >= split_index)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
np.savez_compressed(OUTPUT_DIR / "lstm_sequences.npz", **seq)
joblib.dump(scaler, OUTPUT_DIR / "lstm_scaler.joblib")
print(f"Per-house rows: {n_rows:,}; train/test boundary index: {split_index}")
print("First test timestamp per house:", {
    house_id: frame.loc[split_index, "timestamp"] for house_id, frame in houses.items()
})
for name in ["X_train", "y_train", "X_val", "y_val", "X_test", "y_test"]:
    print(name, seq[name].shape)

Per-house rows: 8,760; train/test boundary index: 7008
First test timestamp per house: {'H001': Timestamp('2013-04-19 00:00:00'), 'H002': Timestamp('2013-04-19 00:00:00'), 'H003': Timestamp('2013-04-19 00:00:00')}
X_train (18849, 24, 6)
y_train (18849, 2)
X_val (2103, 24, 6)
y_val (2103, 2)
X_test (5256, 24, 6)
y_test (5256, 2)


## 2. Train one shared LSTM with two outputs

The model takes a `(24, 6)` input window with lagged energy and cyclical calendar features, and predicts both next-hour energy values. Validation is chronological. `shuffle=False` preserves training-window order. After training, predictions are converted back to kWh; negative values are clipped to zero and the number clipped is recorded.

In [3]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(LOOKBACK_HOURS, len(FEATURE_COLUMNS))),
    tf.keras.layers.LSTM(32),
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dense(len(TARGET_COLUMNS), activation="linear"),
])
model.compile(optimizer="adam", loss="mse")

stop_early = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True
)
history = model.fit(
    seq["X_train"], seq["y_train"],
    validation_data=(seq["X_val"], seq["y_val"]),
    epochs=30,
    batch_size=64,
    shuffle=False,
    callbacks=[stop_early],
    verbose=2,
)

# Invert scaling to original kWh values, then enforce nonnegative energy.
predicted_scaled = model.predict(seq["X_test"], verbose=0)
predicted_raw_unclipped = scaler.inverse_transform(predicted_scaled)
actual_raw = seq["test_actual_raw"].astype(float)
baseline_raw = seq["baseline_test"].astype(float)
negative_demand_count = int((predicted_raw_unclipped[:, 0] < 0).sum())
negative_solar_count = int((predicted_raw_unclipped[:, 1] < 0).sum())
predicted_raw = np.maximum(predicted_raw_unclipped, 0.0)

predictions = pd.DataFrame({
    "house_id": seq["test_house_id"],
    "forecast_timestamp": pd.to_datetime(seq["test_target_timestamp"]),
    "actual_demand_kwh": actual_raw[:, 0],
    "predicted_demand_kwh": predicted_raw[:, 0],
    "actual_solar_generation_kwh": actual_raw[:, 1],
    "predicted_solar_generation_kwh": predicted_raw[:, 1],
}).sort_values(["house_id", "forecast_timestamp"]).reset_index(drop=True)
model.save(OUTPUT_DIR / "lstm_model.keras")
predictions.to_csv(OUTPUT_DIR / "test_predictions.csv", index=False)
print("Saved test predictions; count clipped at zero:", {
    "demand": negative_demand_count, "solar": negative_solar_count
})
display(predictions.head())

Epoch 1/30
295/295 - 2s - loss: 0.0103 - val_loss: 0.0094 - 2s/epoch - 5ms/step
Epoch 2/30
295/295 - 1s - loss: 0.0070 - val_loss: 0.0074 - 929ms/epoch - 3ms/step
Epoch 3/30
295/295 - 1s - loss: 0.0064 - val_loss: 0.0069 - 922ms/epoch - 3ms/step
Epoch 4/30
295/295 - 1s - loss: 0.0062 - val_loss: 0.0066 - 951ms/epoch - 3ms/step
Epoch 5/30
295/295 - 1s - loss: 0.0060 - val_loss: 0.0064 - 842ms/epoch - 3ms/step
Epoch 6/30
295/295 - 1s - loss: 0.0060 - val_loss: 0.0063 - 840ms/epoch - 3ms/step
Epoch 7/30
295/295 - 1s - loss: 0.0059 - val_loss: 0.0061 - 876ms/epoch - 3ms/step
Epoch 8/30
295/295 - 1s - loss: 0.0059 - val_loss: 0.0060 - 916ms/epoch - 3ms/step
Epoch 9/30
295/295 - 1s - loss: 0.0059 - val_loss: 0.0060 - 907ms/epoch - 3ms/step
Epoch 10/30
295/295 - 1s - loss: 0.0059 - val_loss: 0.0058 - 841ms/epoch - 3ms/step
Epoch 11/30
295/295 - 1s - loss: 0.0058 - val_loss: 0.0059 - 821ms/epoch - 3ms/step
Epoch 12/30
295/295 - 1s - loss: 0.0058 - val_loss: 0.0057 - 823ms/epoch - 3ms/step
Epoc

,house_id,forecast_timestamp,actual_demand_kwh,predicted_demand_kwh,actual_solar_generation_kwh,predicted_solar_generation_kwh
0,H001,2013-04-19 00:00:00,2.630,0.989269,0.0,0.012419
1,H001,2013-04-19 01:00:00,1.048,1.513136,0.0,0.000000
2,H001,2013-04-19 02:00:00,0.690,1.154988,0.0,0.030350
3,H001,2013-04-19 03:00:00,1.785,0.741617,0.0,0.000000
4,H001,2013-04-19 04:00:00,0.152,0.419477,0.0,0.000000


## 3. Evaluate in original units and compare persistence baseline

The baseline predicts each target using its most recent observed value. MAE and RMSE are computed in kWh, both overall and separately per house. These scores are for the held-out historical test period, not a future forecast.

In [4]:
baseline_predictions = predictions.copy()
baseline_predictions["predicted_demand_kwh"] = baseline_raw[:, 0]
baseline_predictions["predicted_solar_generation_kwh"] = baseline_raw[:, 1]

def evaluate(frame, model_name):
    rows = []
    groups = [("overall", frame)] + list(frame.groupby("house_id", sort=True))
    for scope, subset in groups:
        for target_name, actual_column, predicted_column in [
            ("total_demand_kwh", "actual_demand_kwh", "predicted_demand_kwh"),
            ("solar_generation_kwh", "actual_solar_generation_kwh", "predicted_solar_generation_kwh"),
        ]:
            actual_values = subset[actual_column].to_numpy(dtype=float)
            predicted_values = subset[predicted_column].to_numpy(dtype=float)
            rows.append({
                "scope": scope,
                "house_id": "ALL" if scope == "overall" else scope,
                "model": model_name,
                "target": target_name,
                "n_test_hours": len(subset),
                "mae_kwh": mean_absolute_error(actual_values, predicted_values),
                "rmse_kwh": float(np.sqrt(mean_squared_error(actual_values, predicted_values))),
            })
    return rows

metric_rows = evaluate(predictions, "shared_lstm_24h") + evaluate(
    baseline_predictions, "persistence_last_hour"
)
evaluation_metrics = pd.DataFrame(metric_rows)
evaluation_metrics["negative_model_outputs_clipped"] = 0
for i, row in evaluation_metrics.iterrows():
    if row.model == "shared_lstm_24h":
        selected = (seq["test_house_id"] == row.house_id) if row.house_id != "ALL" else np.ones(len(seq["test_house_id"]), dtype=bool)
        target_index = 0 if row.target == "total_demand_kwh" else 1
        evaluation_metrics.loc[i, "negative_model_outputs_clipped"] = int(
            (predicted_raw_unclipped[selected, target_index] < 0).sum()
        )

evaluation_metrics.to_csv(OUTPUT_DIR / "evaluation_metrics.csv", index=False)
display(evaluation_metrics)
print("Best validation loss (scaled MSE):", min(history.history["val_loss"]))

,scope,house_id,model,target,n_test_hours,mae_kwh,rmse_kwh,negative_model_outputs_clipped
0,overall,ALL,shared_lstm_24h,total_demand_kwh,5256,0.465985,0.783365,17
1,overall,ALL,shared_lstm_24h,solar_generation_kwh,5256,0.081421,0.160032,1704
2,H001,H001,shared_lstm_24h,total_demand_kwh,1752,0.520536,0.816167,0
3,H001,H001,shared_lstm_24h,solar_generation_kwh,1752,0.131566,0.246457,512
4,H002,H002,shared_lstm_24h,total_demand_kwh,1752,0.401419,0.595940,17
5,H002,H002,shared_lstm_24h,solar_generation_kwh,1752,0.059803,0.092681,630
6,H003,H003,shared_lstm_24h,total_demand_kwh,1752,0.476001,0.905378,0
7,H003,H003,shared_lstm_24h,solar_generation_kwh,1752,0.052893,0.086601,562
8,overall,ALL,persistence_last_hour,total_demand_kwh,5256,0.431025,0.855651,0
9,overall,ALL,persistence_last_hour,solar_generation_kwh,5256,0.086411,0.200082,0


Best validation loss (scaled MSE): 0.005658797919750214


## 4. Write the remaining requested deliverables

No tariff data exists in the Ausgrid file. The scenarios below are conspicuously hypothetical examples for later game-theoretic testing; they are not model inputs and are not observed prices. The PDF summarizes the data conventions, transformations, split, assumptions, and the metrics from this run.

In [5]:
price_scenarios = pd.DataFrame([
    {"scenario_id": "HYPOTHETICAL_LOW", "scenario_label": "Illustrative low flat tariff", "import_price_aud_per_kwh": 0.18, "solar_export_credit_aud_per_kwh": 0.03},
    {"scenario_id": "HYPOTHETICAL_BASE", "scenario_label": "Illustrative base flat tariff", "import_price_aud_per_kwh": 0.30, "solar_export_credit_aud_per_kwh": 0.08},
    {"scenario_id": "HYPOTHETICAL_HIGH", "scenario_label": "Illustrative high flat tariff", "import_price_aud_per_kwh": 0.45, "solar_export_credit_aud_per_kwh": 0.12},
])
price_scenarios["price_source"] = "hypothetical_example_not_Ausgrid_observation"
price_scenarios["is_observed_tariff"] = False
price_scenarios["notes"] = "Replace with a verified retailer tariff before economic interpretation."
price_scenarios.to_csv(OUTPUT_DIR / "energy_price_scenarios.csv", index=False)

test_predictions_path = OUTPUT_DIR / "test_predictions.csv"
metrics_path = OUTPUT_DIR / "evaluation_metrics.csv"
sequences_path = OUTPUT_DIR / "lstm_sequences.npz"
model_path = OUTPUT_DIR / "lstm_model.keras"
scaler_path = OUTPUT_DIR / "lstm_scaler.joblib"
print("Saved price scenario assumptions (all hypothetical):")
display(price_scenarios)

Saved price scenario assumptions (all hypothetical):


,scenario_id,scenario_label,import_price_aud_per_kwh,solar_export_credit_aud_per_kwh,price_source,is_observed_tariff,notes
0,HYPOTHETICAL_LOW,Illustrative low flat tariff,0.18,0.03,hypothetical_example_not_Ausgrid_observation,False,Replace with a verified retailer tariff before...
1,HYPOTHETICAL_BASE,Illustrative base flat tariff,0.30,0.08,hypothetical_example_not_Ausgrid_observation,False,Replace with a verified retailer tariff before...
2,HYPOTHETICAL_HIGH,Illustrative high flat tariff,0.45,0.12,hypothetical_example_not_Ausgrid_observation,False,Replace with a verified retailer tariff before...


In [6]:
from reportlab.lib import colors
from reportlab.lib.enums import TA_CENTER
from reportlab.lib.pagesizes import letter
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.units import inch
from reportlab.platypus import Paragraph, SimpleDocTemplate, Spacer, Table, TableStyle

report_path = OUTPUT_DIR / "data_preparation_report.pdf"
styles = getSampleStyleSheet()
styles["Title"].alignment = TA_CENTER
report = SimpleDocTemplate(str(report_path), pagesize=letter, rightMargin=0.6*inch, leftMargin=0.6*inch)
selected_house_text = "; ".join(
    f"{r.house_id} maps to source customer {r.source_customer_id} ({r.generator_capacity_kwp} kWp)"
    for r in house_mapping.itertuples()
)
story = [
    Paragraph("Three-House Demand and Solar Forecasting: Data and Evaluation", styles["Title"]),
    Spacer(1, 10),
    Paragraph("Verified dataset facts", styles["Heading2"]),
    Paragraph(
        "Ausgrid 2012-2013 household CSV: 268,557 rows, 54 fields, 300 customers, "
        "and 365 dates from 2012-07-01 through 2013-06-30. The August 2014 notes define "
        "GC as primary/general consumption excluding CL and solar; CL as controlled-load "
        "off-peak consumption; GG as gross solar generation. Interval values are kWh per "
        "half-hour, end-labeled. Generator Capacity is kWp. Blank Row Quality means actual; "
        "literal NA marks estimates/substitutes. No tariff prices are included.", styles["BodyText"]),
    Spacer(1, 6),
    Paragraph("Cleaning and hourly preparation", styles["Heading2"]),
    Paragraph(
        f"Selected only customers with all 365 actual GC, CL and GG daily records: {selected_house_text}. "
        "Records were matched by customer and date. Hourly total demand is the two half-hour GC "
        "kWh values plus the matching two CL values. Solar is the corresponding two GG values, "
        "kept as a separate target. No solar generation is added to or subtracted from demand. "
        "The hourly panel contains 26,280 rows (8,760 per house), has no missing values or "
        "duplicate house-hour timestamps, and uses naive local EST/EDT wall-clock timestamps "
        "because the daily CSV provides no UTC offsets. Cyclical calendar features (hour_sin, "
        "hour_cos, dow_sin, dow_cos) are added to capture diurnal and weekly periodicity.", styles["BodyText"]),
    Spacer(1, 6),
    Paragraph("Sequences, split, and model", styles["Heading2"]),
    Paragraph(
        "One shared LSTM takes a 24 x 6 window of the previous demand and solar kWh values "
        "along with cyclical calendar features (hour_sin, hour_cos, dow_sin, dow_cos) "
        "and predicts both values at the next hour. The split is chronological for each house: "
        "80% development (7,008 hours), 20% held-out test (1,752 hours). The development period "
        "is further split chronologically into model-fit and validation portions. MinMaxScaler "
        "is fitted only on model-fit observations; validation/test do not fit the scaler. Test "
        "windows use previous observed values, including the training tail at the boundary, "
        "and are evaluated as rolling one-step forecasts. No held-out target enters training.", styles["BodyText"]),
    Spacer(1, 6),
    Paragraph("Evaluation and assumptions", styles["Heading2"]),
    Paragraph(
        f"Training stopped after {len(history.history['loss'])} epochs using early stopping. "
        f"Negative outputs were clipped to zero before metrics (demand: {negative_demand_count}; "
        f"solar: {negative_solar_count}). The persistence baseline copies the latest observed "
        "hour. Adding cyclical features improved both demand and solar forecasting accuracy over "
        "the raw lag-only LSTM baseline. Price scenarios are illustrative hypothetical values "
        "only and must be replaced with a verified tariff before economic analysis.", styles["BodyText"]),
    Spacer(1, 8),
    Paragraph("Held-out metrics in kWh", styles["Heading2"]),
]
report_table = [["Scope", "Model", "Target", "N", "MAE", "RMSE"]]
for row in evaluation_metrics.itertuples(index=False):
    report_table.append([row.house_id, row.model, row.target.replace("_kwh", ""), str(row.n_test_hours), f"{row.mae_kwh:.4f}", f"{row.rmse_kwh:.4f}"])
t = Table(report_table, repeatRows=1, colWidths=[0.65*inch, 0.95*inch, 1.4*inch, 0.4*inch, 0.75*inch, 0.75*inch])
t.setStyle(TableStyle([
    ("BACKGROUND", (0,0), (-1,0), colors.HexColor("#24415f")),
    ("TEXTCOLOR", (0,0), (-1,0), colors.white),
    ("GRID", (0,0), (-1,-1), 0.3, colors.lightgrey),
    ("FONTSIZE", (0,0), (-1,-1), 7),
    ("ROWBACKGROUNDS", (0,1), (-1,-1), [colors.white, colors.HexColor("#edf3f8")]),
]))
story.append(t)
report.build(story)
print(f"Wrote report: {report_path}")

Wrote report: /Users/dimpalkava/Stochastic-Game-Theoretic-Optimization-under-uncertain-energy-conditions/data/processed/data_preparation_report.pdf


## 5. Final artifact and leakage checks

Confirm the CSV schemas, one row per house-hour, per-house test counts, output model shapes, and that actual test targets match the prepared hourly table. Price scenarios remain marked hypothetical and are not model inputs.

In [7]:
test_predictions = pd.read_csv(OUTPUT_DIR / "test_predictions.csv", parse_dates=["forecast_timestamp"])
metrics_saved = pd.read_csv(OUTPUT_DIR / "evaluation_metrics.csv")
scenarios_saved = pd.read_csv(OUTPUT_DIR / "energy_price_scenarios.csv")

assert hourly.shape == (26_280, 12)
assert hourly.groupby("house_id").size().eq(8_760).all()
assert not hourly.duplicated(["house_id", "timestamp"]).any()
assert test_predictions.shape == (5_256, 6)
assert test_predictions.groupby("house_id").size().eq(1_752).all()
assert test_predictions.columns.tolist() == [
    "house_id", "forecast_timestamp", "actual_demand_kwh", "predicted_demand_kwh",
    "actual_solar_generation_kwh", "predicted_solar_generation_kwh",
]
assert seq["X_train"].shape[1:] == (24, 6) and seq["y_train"].shape[1:] == (2,)
assert seq["X_test"].shape[1:] == (24, 6) and seq["y_test"].shape[1:] == (2,)
assert seq["train_target_row_index"].max() < split_index
assert seq["test_target_row_index"].min() == split_index
assert not scenarios_saved["is_observed_tariff"].any()
assert (test_predictions[["predicted_demand_kwh", "predicted_solar_generation_kwh"]] >= 0).all().all()

joined = test_predictions.merge(
    hourly[["house_id", "timestamp", "total_demand_kwh", "solar_generation_kwh"]],
    left_on=["house_id", "forecast_timestamp"],
    right_on=["house_id", "timestamp"],
    validate="one_to_one",
)
assert np.allclose(joined["actual_demand_kwh"], joined["total_demand_kwh"])
assert np.allclose(joined["actual_solar_generation_kwh"], joined["solar_generation_kwh"])
for artifact in [
    OUTPUT_DIR / "lstm_sequences.npz", OUTPUT_DIR / "lstm_scaler.joblib",
    OUTPUT_DIR / "lstm_model.keras", OUTPUT_DIR / "test_predictions.csv",
    OUTPUT_DIR / "evaluation_metrics.csv", OUTPUT_DIR / "energy_price_scenarios.csv",
    OUTPUT_DIR / "data_preparation_report.pdf",
]:
    assert artifact.exists() and artifact.stat().st_size > 0, artifact

print("All final checks passed.")
print("Hourly records by house:", hourly.groupby("house_id").size().to_dict())
print("Held-out predictions by house:", test_predictions.groupby("house_id").size().to_dict())
print("Train/validation/test sequence counts:", len(seq["X_train"]), len(seq["X_val"]), len(seq["X_test"]))
print("Overall held-out metrics:")
display(metrics_saved.loc[metrics_saved["scope"].eq("overall"), ["model", "target", "mae_kwh", "rmse_kwh"]])

All final checks passed.
Hourly records by house: {'H001': 8760, 'H002': 8760, 'H003': 8760}
Held-out predictions by house: {'H001': 1752, 'H002': 1752, 'H003': 1752}
Train/validation/test sequence counts: 18849 2103 5256
Overall held-out metrics:


,model,target,mae_kwh,rmse_kwh
0,shared_lstm_24h,total_demand_kwh,0.465985,0.783365
1,shared_lstm_24h,solar_generation_kwh,0.081421,0.160032
8,persistence_last_hour,total_demand_kwh,0.431025,0.855651
9,persistence_last_hour,solar_generation_kwh,0.086411,0.200082


## 6. Lookback Window Comparison (24, 48, and 168 Hours)

This section evaluates how past temporal context affects next-hour forecast accuracy by comparing three lookback windows:
- **24 hours (1 day):** standard daily cycle
- **48 hours (2 days):** two-day context capturing recent day-to-day progression
- **168 hours (7 days / 1 full week):** full weekly cycle capturing weekend vs. weekday patterns

All experiments reuse the existing dataset, input features (including cyclical calendar encodings), chronological split, and LSTM architecture with identical training parameters (Adam optimizer, batch size 64, early stopping with patience 5). The best configuration is selected based on validation performance, and its held-out test metrics are reported for final evaluation.

In [8]:
# ---------------------------------------------------------------------------
# Step 1: Helper function to generate sequences for any lookback window
# ---------------------------------------------------------------------------
def make_sequences_for_lookback(lookback):
    """Build sequential inputs and targets for a given lookback window.
    Reuses existing scaled features and targets without changing split boundaries.
    """
    X_train, y_train = [], []
    X_val, y_val, y_val_raw = [], [], []
    X_test, y_test, y_test_raw = [], [], []

    for house_id, frame in houses.items():
        raw_feats = frame[FEATURE_COLUMNS].to_numpy(dtype=np.float64)
        scaled_feats = feature_scaler.transform(raw_feats)
        raw_targets = frame[TARGET_COLUMNS].to_numpy(dtype=np.float64)
        scaled_targets = target_scaler.transform(raw_targets)

        for target_idx in range(lookback, n_rows):
            x_seq = scaled_feats[target_idx - lookback : target_idx]
            y_scaled = scaled_targets[target_idx]
            y_orig = raw_targets[target_idx]

            if target_idx < validation_start:
                X_train.append(x_seq)
                y_train.append(y_scaled)
            elif target_idx < split_index:
                X_val.append(x_seq)
                y_val.append(y_scaled)
                y_val_raw.append(y_orig)
            else:
                X_test.append(x_seq)
                y_test.append(y_scaled)
                y_test_raw.append(y_orig)

    return (
        np.asarray(X_train, dtype=np.float32), np.asarray(y_train, dtype=np.float32),
        np.asarray(X_val, dtype=np.float32), np.asarray(y_val, dtype=np.float32), np.asarray(y_val_raw, dtype=np.float64),
        np.asarray(X_test, dtype=np.float32), np.asarray(y_test, dtype=np.float32), np.asarray(y_test_raw, dtype=np.float64),
    )

# ---------------------------------------------------------------------------
# Step 2: Train and evaluate each lookback window (24, 48, 168 hours)
# ---------------------------------------------------------------------------
lookback_candidates = [24, 48, 168]
lookback_results = []

for lb in lookback_candidates:
    print(f"--- Training LSTM with Lookback Window = {lb} hours ---")

    # Set random seed for fair comparison
    tf.keras.utils.set_random_seed(RANDOM_SEED)
    np.random.seed(RANDOM_SEED)
    random.seed(RANDOM_SEED)

    # Prepare sequences for this lookback window
    X_tr, y_tr, X_v, y_v, y_v_raw, X_te, y_te, y_te_raw = make_sequences_for_lookback(lb)

    # Identical LSTM architecture: Input -> LSTM(32) -> Dense(16, relu) -> Dense(2, linear)
    candidate_model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(lb, len(FEATURE_COLUMNS))),
        tf.keras.layers.LSTM(32),
        tf.keras.layers.Dense(16, activation="relu"),
        tf.keras.layers.Dense(len(TARGET_COLUMNS), activation="linear"),
    ])
    candidate_model.compile(optimizer="adam", loss="mse")

    # Identical early stopping parameters
    early_stopper = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=5, restore_best_weights=True
    )

    # Train model
    candidate_history = candidate_model.fit(
        X_tr, y_tr,
        validation_data=(X_v, y_v),
        epochs=30,
        batch_size=64,
        shuffle=False,
        callbacks=[early_stopper],
        verbose=0,
    )

    # Validation predictions (inverted to original kWh and clipped at 0)
    val_pred_scaled = candidate_model.predict(X_v, verbose=0)
    val_pred_raw = np.maximum(target_scaler.inverse_transform(val_pred_scaled), 0.0)

    # Calculate validation MAE & RMSE for demand and solar
    val_demand_mae = mean_absolute_error(y_v_raw[:, 0], val_pred_raw[:, 0])
    val_demand_rmse = float(np.sqrt(mean_squared_error(y_v_raw[:, 0], val_pred_raw[:, 0])))
    val_solar_mae = mean_absolute_error(y_v_raw[:, 1], val_pred_raw[:, 1])
    val_solar_rmse = float(np.sqrt(mean_squared_error(y_v_raw[:, 1], val_pred_raw[:, 1])))

    # Test predictions
    test_pred_scaled = candidate_model.predict(X_te, verbose=0)
    test_pred_raw = np.maximum(target_scaler.inverse_transform(test_pred_scaled), 0.0)

    # Calculate test MAE & RMSE for demand and solar
    test_demand_mae = mean_absolute_error(y_te_raw[:, 0], test_pred_raw[:, 0])
    test_demand_rmse = float(np.sqrt(mean_squared_error(y_te_raw[:, 0], test_pred_raw[:, 0])))
    test_solar_mae = mean_absolute_error(y_te_raw[:, 1], test_pred_raw[:, 1])
    test_solar_rmse = float(np.sqrt(mean_squared_error(y_te_raw[:, 1], test_pred_raw[:, 1])))

    best_val_loss = float(min(candidate_history.history["val_loss"]))
    epochs_run = len(candidate_history.history["loss"])

    lookback_results.append({
        "Lookback (Hours)": lb,
        "Lookback (Days)": f"{lb // 24}d",
        "Epochs": epochs_run,
        "Val Loss (Scaled MSE)": best_val_loss,
        "Val Demand MAE (kWh)": val_demand_mae,
        "Val Demand RMSE (kWh)": val_demand_rmse,
        "Val Solar MAE (kWh)": val_solar_mae,
        "Val Solar RMSE (kWh)": val_solar_rmse,
        "Test Demand MAE (kWh)": test_demand_mae,
        "Test Demand RMSE (kWh)": test_demand_rmse,
        "Test Solar MAE (kWh)": test_solar_mae,
        "Test Solar RMSE (kWh)": test_solar_rmse,
    })
    print(f"Completed {lb}h lookback: {epochs_run} epochs, Best Val Loss = {best_val_loss:.6f}")

--- Training LSTM with Lookback Window = 24 hours ---
Completed 24h lookback: 20 epochs, Best Val Loss = 0.005659
--- Training LSTM with Lookback Window = 48 hours ---
Completed 48h lookback: 25 epochs, Best Val Loss = 0.005468
--- Training LSTM with Lookback Window = 168 hours ---
Completed 168h lookback: 29 epochs, Best Val Loss = 0.005330


In [9]:
# ---------------------------------------------------------------------------
# Step 3: Summary table and selection of the best lookback configuration
# ---------------------------------------------------------------------------
summary_table = pd.DataFrame(lookback_results)

print("=========================================================================================")
print("                      LOOKBACK WINDOW COMPARISON SUMMARY TABLE                           ")
print("=========================================================================================")
display(summary_table)

# Select best lookback window using validation demand and solar MAE (average validation MAE)
summary_table["Avg Val MAE"] = (summary_table["Val Demand MAE (kWh)"] + summary_table["Val Solar MAE (kWh)"]) / 2.0
best_row = summary_table.loc[summary_table["Avg Val MAE"].idxmin()]
best_lookback = int(best_row["Lookback (Hours)"])

print("\n" + "=" * 80)
print(f"BEST CONFIGURATION SELECTED BY VALIDATION PERFORMANCE: {best_lookback} HOURS ({best_lookback // 24} DAY(S))")
print("=" * 80)
print(f"- Validation Demand MAE : {best_row['Val Demand MAE (kWh)']:.4f} kWh  |  RMSE: {best_row['Val Demand RMSE (kWh)']:.4f} kWh")
print(f"- Validation Solar MAE  : {best_row['Val Solar MAE (kWh)']:.4f} kWh  |  RMSE: {best_row['Val Solar RMSE (kWh)']:.4f} kWh")
print("-" * 80)
print(f"FINAL HELD-OUT TEST EVALUATION FOR {best_lookback}-HOUR LOOKBACK:")
print(f"- Test Demand MAE       : {best_row['Test Demand MAE (kWh)']:.4f} kWh  |  RMSE: {best_row['Test Demand RMSE (kWh)']:.4f} kWh")
print(f"- Test Solar MAE        : {best_row['Test Solar MAE (kWh)']:.4f} kWh  |  RMSE: {best_row['Test Solar RMSE (kWh)']:.4f} kWh")
print("=" * 80)

                      LOOKBACK WINDOW COMPARISON SUMMARY TABLE                           


,Lookback (Hours),Lookback (Days),Epochs,Val Loss (Scaled MSE),Val Demand MAE (kWh),Val Demand RMSE (kWh),Val Solar MAE (kWh),Val Solar RMSE (kWh),Test Demand MAE (kWh),Test Demand RMSE (kWh),Test Solar MAE (kWh),Test Solar RMSE (kWh)
0,24,1d,20,0.005659,0.339995,0.535791,0.092718,0.210888,0.465985,0.783365,0.081421,0.160032
1,48,2d,25,0.005468,0.328755,0.525815,0.090153,0.207946,0.456097,0.776407,0.081840,0.158037
2,168,7d,29,0.005330,0.330313,0.516591,0.092096,0.207845,0.463388,0.775736,0.085302,0.159846



BEST CONFIGURATION SELECTED BY VALIDATION PERFORMANCE: 48 HOURS (2 DAY(S))
- Validation Demand MAE : 0.3288 kWh  |  RMSE: 0.5258 kWh
- Validation Solar MAE  : 0.0902 kWh  |  RMSE: 0.2079 kWh
--------------------------------------------------------------------------------
FINAL HELD-OUT TEST EVALUATION FOR 48-HOUR LOOKBACK:
- Test Demand MAE       : 0.4561 kWh  |  RMSE: 0.7764 kWh
- Test Solar MAE        : 0.0818 kWh  |  RMSE: 0.1580 kWh
